# Ejercicio AirBnB: Extracción de entidades

En este cuaderno vamos a trabajar con un dataset de AirBnB de la ciudad de Oporto. Se puede encontrar más información sobre el dataset y otras implementaciones en [Porto](https://github.com/Vasallo94/Porto). 

El dataset contiene información sobre las características de las viviendas, su localización, el precio, el número de comentarios, etc.

El objetivo de este ejercicio es poder analizar los comentarios de los usuarios mediante LLMs y poder extraer información relevante de los mismos para su análisis posterior.

## Primera parte

Vamos a utilizar el archivo listings1_cleaned.csv que contiene información sobre las viviendas de AirBnB en Oporto.

*He modificado un poco la celda para no meter los Key directamente aqui*

In [1]:
import os
from dotenv import load_dotenv
import pandas as pd
from google import genai
from IPython.display import display, Markdown

# Cargar variables del archivo .env
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")


#### Cargamos el dataset a usar

In [3]:
data = pd.read_csv("listings1_cleaned.csv")
data


,listing_id,listing_url,picture_url,name,description,host_id,host_name,host_since,host_response_rate,host_acceptance_rate,...,review_scores_rating,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,reviews_per_month,availability_365,has_availability,last_review,geographical_group
0,2.905900e+04,https://www.airbnb.com/rooms/29059,https://a0.muscache.com/pictures/736399/fa6c31...,Lovely studio Quartier Latin,CITQ 267153<br />Lovely studio with 1 closed r...,125031.0,Maryline,2010-05-14,100.000000,98.000000,...,4.670000,4.620000,4.810000,4.770000,4.820000,2.69000,302.0,t,2024-03-16,Central
1,2.906100e+04,https://www.airbnb.com/rooms/29061,https://a0.muscache.com/pictures/9e59d417-4b6a...,Maison historique - Quartier Latin,Lovely historic house with plenty of period ch...,125031.0,Maryline,2010-05-14,100.000000,98.000000,...,4.730000,4.660000,4.880000,4.810000,4.870000,0.88000,348.0,t,2024-02-19,Central
2,3.630100e+04,https://www.airbnb.com/rooms/36301,https://a0.muscache.com/pictures/26c20544-475f...,Romantic & peaceful Plateau loft,"Enjoy the best of Montreal in this romantic, ...",381468.0,Sylvie,2011-02-07,94.000000,80.000000,...,4.860000,4.860000,4.920000,4.900000,4.880000,0.47000,81.0,t,2024-01-07,Central
3,3.811800e+04,https://www.airbnb.com/rooms/38118,https://a0.muscache.com/pictures/213997/763ec1...,Beautiful room with a balcony in front of a parc,Nearest metro Papineau.,163569.0,M.,2010-07-11,78.000000,0.000000,...,4.500000,4.250000,4.810000,4.810000,4.630000,0.10000,299.0,t,2022-08-29,Central
4,5.047900e+04,https://www.airbnb.com/rooms/50479,https://a0.muscache.com/pictures/miso/Hosting-...,L'Arcade Douce,The appartement is sunny and ideally situated ...,231694.0,Noemie,2010-09-11,100.000000,100.000000,...,4.950000,4.940000,4.970000,4.980000,4.840000,1.60000,0.0,t,2024-03-18,South
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8135,1.116753e+18,https://www.airbnb.com/rooms/1116753448972266015,https://a0.muscache.com/pictures/miso/Hosting-...,"Logement ,1 chambre 04 personnes",If you are looking for a beautiful cozy and we...,429774011.0,Ammar,2021-10-31,96.922347,89.203748,...,4.715698,4.692737,4.814562,4.812302,4.768682,1.70759,257.0,t,NaN,North
8136,1.116820e+18,https://www.airbnb.com/rooms/1116820020797670135,https://a0.muscache.com/pictures/miso/Hosting-...,Superbe maison à Montréal,Beautiful entire place in the heart of Montreal,126764590.0,Samuel,2017-04-20,96.922347,89.203748,...,4.715698,4.692737,4.814562,4.812302,4.768682,1.70759,269.0,t,NaN,East
8137,1.117028e+18,https://www.airbnb.com/rooms/1117028063118176710,https://a0.muscache.com/pictures/miso/Hosting-...,Luxurious 2BR - Downtown Montreal,Feel at home at these brand new construction C...,159008278.0,Melissa,2017-11-16,100.000000,100.000000,...,4.715698,4.692737,4.814562,4.812302,4.768682,1.70759,156.0,t,NaN,Central
8138,1.117273e+18,https://www.airbnb.com/rooms/1117273423317641658,https://a0.muscache.com/pictures/miso/Hosting-...,Magnifique appart Plateau,"Fully renovated condo style apartment, beautif...",214303569.0,Jean-Georges,2018-09-08,100.000000,93.000000,...,4.715698,4.692737,4.814562,4.812302,4.768682,1.70759,192.0,t,NaN,Central


1. Filtra el dataset con una función que seleccione las 10/20 reseñas con mayor longitud (de string) en la columna descripción (las que consideraríamos las más 'relevantes')
2. Elabora un prompt de tal forma que un LLM sea capaz de extraer un json con las siguientes entidades de las entradas de la tabla filtrada:
```json
{
    "name": str,
    "location": str,
    "main_characteristics": str,
    "type": str,
    "size": str,
    "capacity": str,
    "key_amenities": list,
    "proximity_highlights": list,
}
```
3. Puedes elegir el proveedor que quieras, Gemini de Google o cualquiera de los modelos de Groq

In [4]:
import pandas as pd
import re

# Cargar dataset
data = pd.read_csv("listings1_cleaned.csv")

# Limpiar HTML básico
def clean_html(text):
    if isinstance(text, str):
        return re.sub(r"<.*?>", " ", text)
    return ""

data["description_clean"] = data["description"].astype(str).apply(clean_html)

# Añadir longitud
data["description_length"] = data["description_clean"].apply(len)

# Seleccionar las 20 más largas para groq
top_reviews = data.sort_values(by="description_length", ascending=False).head(20)

top_reviews[["name", "description_clean", "description_length"]]

,name,description_clean,description_length
1017,Modern Studio - Close to Downtown and Old Mont...,You will enjoy sleeping in our soft and comfor...,1000
1035,Charming Studio - Next to downtown,You will enjoy sleeping in our soft and comfor...,1000
1132,Studio Apt-Free Parking Near Downtown/Old Mont...,You will enjoy sleeping in our soft and comfor...,1000
1352,Business Ready❤Dwtwn Conference Palais de Cong...,"Composition: bathroom, WC, bedroom / living ro...",995
835,Luxury Historic Montreal 3 Bdrm w/Deck. # 296183,Indulge in the luxury of classic Montreal arch...,985
77,Bohemian Loft Retreat in Montreal’s Old Port,The old-world charm in this spacious artist lo...,980
1137,"Appartement aussi spacieux que confortable, à ...",Located on the 2nd floor of a duplex located o...,970
174,joli appartement économique,Large apartment located on the island of Montr...,970
2733,"une chambre de 20m2, idéal pour les voyageurs","Large studio next to the subway, fully furnish...",965
1202,Mile End Condo with City Views from the Roof Deck,Greet the day as morning light fills a chic an...,955


In [5]:
extraction_prompt = """
Eres un asistente experto en análisis de alojamientos turísticos de OPORTO.

A partir de la descripción de un alojamiento de AirBnB, extrae la información solicitada y devuélvela EXCLUSIVAMENTE en formato JSON válido.

Estructura requerida:
{
    "name": str,
    "location": str,
    "main_characteristics": str,
    "type": str,
    "size": str,
    "capacity": str,
    "key_amenities": list,
    "proximity_highlights": list
}

Instrucciones:
- Usa la columna 'name' como nombre del alojamiento.
- Usa 'neighbourhood' como ubicación si está disponible; si no, infiere la ubicación desde la descripción.
- 'type' debe venir de 'property_type' o 'room_type'.
- 'size' puede inferirse de bedrooms/beds/bathrooms.
- 'capacity' debe venir de 'accommodates'.
- 'key_amenities' debe ser una lista de comodidades relevantes extraídas de la columna 'amenities'.
- 'proximity_highlights' debe incluir lugares cercanos, transporte, zonas de interés, etc., inferidos del texto.
- Si algún dato no aparece, deja "" o una lista vacía.
- NO incluyas explicaciones, solo el JSON final.

Descripción del alojamiento:
"""


#### Llamamos al modelo de Groq

In [6]:
from groq import Groq
import json

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Selecciona las 20 reseñas más largas (o las que quieras)
top_reviews_groq = top_reviews.head(20)


def clean_json_output(text):
    """
    Limpia bloques tipo ```json ... ``` o ``` ... ```
    para dejar solo el JSON puro.
    """
    text = text.strip()

    # Si empieza con ```
    if text.startswith("```"):
        parts = text.split("```")
        if len(parts) >= 2:
            text = parts[1].strip()

    # Si empieza con "json" o "JSON"
    if text.lower().startswith("json"):
        text = text[4:].strip()

    # Quitar ``` finales si quedaron
    if text.endswith("```"):
        text = text[:-3].strip()

    return text


results = []

for idx, row in top_reviews_groq.iterrows():
    description = row["description_clean"]

    full_prompt = extraction_prompt + description

    completion = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "Eres un extractor de información en JSON."},
            {"role": "user", "content": full_prompt}
        ]
    )

    raw = completion.choices[0].message.content
    cleaned = clean_json_output(raw)

    try:
        parsed = json.loads(cleaned)
    except Exception:
        parsed = {
            "error": "JSON inválido",
            "raw_output": raw,
            "cleaned_attempt": cleaned
        }

    results.append(parsed)

results[:3]


[{'name': '',
  'location': 'cerca de Promenade Masson, Downtown y Old Montreal',
  'main_characteristics': 'cama cómoda, baño completo, cocina equipada',
  'type': '',
  'size': '1 cama, 1 baño',
  'capacity': '',
  'key_amenities': ['cama cómoda',
   'baño completo',
   'cocina equipada',
   'tienda de conveniencia',
   "restaurante McDonald's"],
  'proximity_highlights': ['Promenade Masson',
   'tienda de comestibles Maxi',
   'estación de autobús',
   'metro Iberville',
   'metro Frontenac',
   'Biodome',
   'Estadio Olímpico',
   'Downtown',
   'Old Montreal']},
 {'name': '',
  'location': 'cerca de promenade Masson, downtown y old Montreal',
  'main_characteristics': 'cama Quinn size, baño equipado, cocina operativa',
  'type': '',
  'size': '1 cama Quinn size, baño',
  'capacity': '',
  'key_amenities': ['cama cómoda',
   'baño equipado',
   'cocina operativa',
   'Wi-Fi',
   'towels'],
  'proximity_highlights': ['Maxi (tienda de comestibles)',
   "McDonald's",
   'promenade Mas

#### Llamamos al modelo de Gemini

Por el uso limitado del API Key de Gemini se dificulta poder obtener de forma correcta la informacion.

In [7]:
import json
from google import genai

# Inicializar cliente Gemini con la API key cargada desde .env
gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

# Seleccionamos las 5 reseñas más largas (ya calculadas previamente)
top_reviews_gemini = top_reviews.head(5)

results_gemini = []

for idx, row in top_reviews_gemini.iterrows():
    description = row["description_clean"]

    full_prompt = extraction_prompt + description

    # Llamada al modelo Gemini
    response = gemini_client.models.generate_content(
        model="gemini-2.5-flash",
        contents=full_prompt
    )

    raw = (response.text or "").strip()
    cleaned = clean_json_output(raw)

    # Intentar parsear JSON
    try:
        parsed = json.loads(cleaned)
    except Exception:
        parsed = {
            "error": "JSON inválido",
            "raw_output": raw,
            "cleaned_attempt": cleaned
        }

    results_gemini.append(parsed)

results_gemini[:3]

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Your prepayment credits are depleted. Please go to AI Studio at https://ai.studio/projects to manage your project and billing. Learn more at https://ai.google.dev/gemini-api/docs/billing#prepay. ', 'status': 'RESOURCE_EXHAUSTED'}}

## Parte 2

Realiza un análisis de los comentarios de los apartamentos con un LLM y extrae información relevante de los mismos.

Por ejemplo, puedes analizar los comentarios y extraer información sobre la limpieza, la ubicación, la relación calidad-precio, el sentimiento del comentario, etc. 

Igual que la parte 1 pero con un segundo dataset y con formato de salida a tu gusto

In [8]:
comentarios = pd.read_csv("Airbnb_reviews_5000.csv")
comentarios

,name,host_id,host_name,date,reviewer_id,reviewer_name,comments,language
0,LUXURY apartment t3 oporto antas,37249350.0,Joaquim,2018-01-30,156292241.0,Silvia,Acomodação espaçosa e bonita. O Joaquim e sua ...,pt
1,Aida's Haven | Room&PrivateBath | St. Catarina,38365612.0,Alexandra,2019-12-15,283158910.0,Tierry Dayan,"A anfitriã Alexandra foi muito simpática, pre...",pt
2,Maritime Inspiration - One Bedroom Beach Apart...,147469727.0,Susana,2018-08-12,173756309.0,Julio,"Apartamento muy bien comunicado, aunque nos es...",es
3,Central charming Top floor - nice views,26222276.0,A.Maria,2016-05-28,11705876.0,Rafael,"Adosinda was kind, showed the apartment, set u...",en
4,Ribeira Oporto Apartment II (Renewed 2021),35057317.0,João,2016-06-02,38269559.0,Anais,"Un appartement au coeur de porto, très grand a...",fr
...,...,...,...,...,...,...,...,...
4995,Marquês's House - Estúdio ao Marquês,93991335.0,Conceição,2017-09-10,13710198.0,Sandra,Merci pour le chaleureux accueil et tous les b...,fr
4996,M1.4 | Surf Beach Matosinhos | Porto,39551356.0,Porto Je T'Aime,2018-08-04,204048434.0,Louis,Toutes les qualités recherchées dans ce type d...,fr
4997,Kitchnette Studio in Porto's Downtown II,1651078.0,Lurdes,2022-06-19,68564036.0,Angie,"Its a lovely little studio, a green oasis, rig...",en
4998,MyLoft-Charming Apartment!!! - Metro Bolhão,34348897.0,Carla,2022-06-23,283375850.0,Angelika,Carla ist wirklich eine sehr tolle Gastgeberin...,de


*Antes de calcular la longitud, convertimos todo a string y reemplaza NaN por cadena vacía.*

In [10]:
# Asegurar que no haya NaN y todo sea string
comentarios["comments"] = comentarios["comments"].fillna("").astype(str)

# Calcular longitud
comentarios["comment_length"] = comentarios["comments"].apply(len)

# Seleccionar los 20 más largos
top_comments = comentarios.sort_values(by="comment_length", ascending=False).head(20)

top_comments[["comments", "comment_length"]]

,comments,comment_length
2000,This WOULD have been a 5 star review BUT:<br/>...,2976
3876,Two of us stayed in the apartment for a week i...,2392
2456,"The location of the flat is great, few minutes...",2226
1908,Amazing views of the Douro River with Porto on...,2011
1636,Não recomendo! De todo! <br/>Este Airbnb é no ...,1928
205,Fomos surpreendidos por um delicioso vinho do ...,1846
1963,The apartment was alright for its price range....,1817
4992,Apartamento bien distribuido y bien surtido. M...,1808
4707,On visitait Porto pour la première fois avec m...,1795
1957,We very much enjoyed our stay at Rui’s apartme...,1781


#### Seleccionamos los 20 comentarios más largos del DataSet

In [11]:
comentarios["comment_length"] = comentarios["comments"].astype(str).apply(len)
top_comments = comentarios.sort_values(by="comment_length", ascending=False).head(20)

top_comments[["comments", "comment_length"]]

,comments,comment_length
2000,This WOULD have been a 5 star review BUT:<br/>...,2976
3876,Two of us stayed in the apartment for a week i...,2392
2456,"The location of the flat is great, few minutes...",2226
1908,Amazing views of the Douro River with Porto on...,2011
1636,Não recomendo! De todo! <br/>Este Airbnb é no ...,1928
205,Fomos surpreendidos por um delicioso vinho do ...,1846
1963,The apartment was alright for its price range....,1817
4992,Apartamento bien distribuido y bien surtido. M...,1808
4707,On visitait Porto pour la première fois avec m...,1795
1957,We very much enjoyed our stay at Rui’s apartme...,1781


#### Prompt para el análisis de los comentarios

In [12]:
review_prompt = """
Eres un analista experto en reseñas de alojamientos turísticos.

A partir del siguiente comentario de un huésped, extrae la siguiente información en formato JSON:

{
    "sentimiento_general": str,        # positivo / negativo / neutro
    "limpieza": str,                   # valoración inferida
    "ubicacion": str,                  # valoración inferida
    "calidad_precio": str,             # valoración inferida
    "aspectos_positivos": list,        # lista de puntos positivos
    "aspectos_negativos": list,        # lista de puntos negativos
    "resumen": str                     # resumen de 1-2 líneas
}

No incluyas explicaciones. Devuelve solo JSON válido.

Comentario:
"""

#### Hacemos el llamado al modelo de Groq

In [14]:
from groq import Groq
import json
import re

# Cliente Groq
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Limpiador robusto de JSON (extrae solo el bloque entre llaves)
def clean_json_output(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        return match.group(0).strip()
    return text.strip()

results_reviews = []

for idx, row in top_comments.iterrows():
    comment = row["comments"]

    full_prompt = review_prompt + comment

    completion = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "Eres un extractor de información en JSON."},
            {"role": "user", "content": full_prompt}
        ]
    )

    raw = completion.choices[0].message.content.strip()
    cleaned = clean_json_output(raw)

    try:
        parsed = json.loads(cleaned)
    except Exception:
        parsed = {
            "error": "JSON inválido",
            "raw_output": raw,
            "cleaned_attempt": cleaned
        }

    results_reviews.append(parsed)

results_reviews[:3]



[{'sentimiento_general': 'negativo',
  'limpieza': 'neutro',
  'ubicacion': 'neutro',
  'calidad_precio': 'negativo',
  'aspectos_positivos': ['espacio hermoso',
   'bien equipado',
   'lavadora/secadora funcionando'],
  'aspectos_negativos': ['falta de mantenimiento',
   'no se proporciona jabón para la ducha o champú',
   'no se proporciona jabón para la lavadora',
   'mala gestión de la empresa',
   'proceso de check-in inconveniente'],
  'resumen': 'Un alojamiento con gran potencial, pero arruinado por la mala gestión de la empresa y la falta de mantenimiento y servicios básicos.'},
 {'sentimiento_general': 'positivo',
  'limpieza': 'buena',
  'ubicacion': 'buena',
  'calidad_precio': 'buena',
  'aspectos_positivos': ['Buena ubicación',
   'Apartamento recién reformado',
   'Fábio es un buen anfitrión',
   'Equipo de cocina completo',
   'Wi-Fi y TV funcionan bien',
   'Buena relación calidad-precio'],
  'aspectos_negativos': ['Bañera pequeña',
   'Colchón duro',
   'Presión del ag

#### Exportamos el resultado del analisis

In [15]:
import pandas as pd

df_reviews = pd.DataFrame(results_reviews)
df_reviews["comentario_original"] = top_comments["comments"].values

df_reviews.to_csv("analisis_comentarios_airbnb.csv", index=False)

df_reviews

,sentimiento_general,limpieza,ubicacion,calidad_precio,aspectos_positivos,aspectos_negativos,resumen,comentario_original
0,negativo,neutro,neutro,negativo,"[espacio hermoso, bien equipado, lavadora/seca...","[falta de mantenimiento, no se proporciona jab...","Un alojamiento con gran potencial, pero arruin...",This WOULD have been a 5 star review BUT:<br/>...
1,positivo,buena,buena,buena,"[Buena ubicación, Apartamento recién reformado...","[Bañera pequeña, Colchón duro, Presión del agu...","Apartamento limpio y bien ubicado, con un anfi...",Two of us stayed in the apartment for a week i...
2,positivo,neutro,positivo,positivo,"[Ubicación excelente, Anfitrión amable y ayuda...","[Cortina de la ducha no funciona bien, Falta d...",Alojamiento con excelente ubicación y anfitrió...,"The location of the flat is great, few minutes..."
3,positivo,no mencionada,excelente,buena,"[vistas impresionantes del río Duero, buena ub...",[baño reducido y accesible solo desde el balcó...,Apartamento con vistas impresionantes y excele...,Amazing views of the Douro River with Porto on...
4,negativo,mala,mala,mala,[],"[ubicación poco segura, problemas con la llave...",No recomendado debido a problemas con la ubica...,Não recomendo! De todo! <br/>Este Airbnb é no ...
5,neutro,buena,excelente,aceptable,"[delicioso vino de bienvenida, cama confortabl...","[banheira desactivada, falta de iluminación y ...","Alojamiento bonito y bien ubicado, pero con al...",Fomos surpreendidos por um delicioso vinho do ...
6,neutro,negativa,no mencionada,aceptable,"[buena comunicación con el anfitrión, cocina b...","[ducha con problemas de desagüe, falta de cale...","Alojamiento con potencial, pero con varios pro...",The apartment was alright for its price range....
7,positivo,no mencionada,regular,buena,"[apartamento bien distribuido, incluye parking...",[debe recoger las llaves en el centro de Porto...,"Apartamento cómodo y bien equipado, aunque la ...",Apartamento bien distribuido y bien surtido. M...
8,positivo,buena,excelente,buena,"[Ubicación cerca de la estación de metro, Mais...","[Ruido del autobús en la calle, Falta de bols ...",Un apartamento excelente en una ubicación inme...,On visitait Porto pour la première fois avec m...
9,positivo,buena,excelente,buena,"[apartamento limpio y bien decorado, ubicación...","[falta de privacidad en los baños, diseño de l...","Apartamento céntrico y bien equipado, pero con...",We very much enjoyed our stay at Rui’s apartme...


### Resumen

Basandome en las 20 reseñas mas detalladas, recomendaria estas ofertas de Airbnb en Oporto por su ubicacion priviligeada y limpieza, sim embargo se debe de prestar atencion a las quejas sobre el mantenimiento y la dispinibilidad de utensilios basicos en la cocina, por si le es de bastante interes este aspecto.

#### **Nota:**
Tener en cuenta que por temas de disponibilidad y de uso de las API Key de Gemini y de Groq, solo se pudo obtener la informacion con Groq, ya que la de Gemini al tratar de hacer las consultas siempre se terminaban consumiendo los llamados por minutos y dificultaba bastante la obtencion de los datos con este modelo.